# UAV Continual Learning — chạy TẤT CẢ trên Colab (không cần thẻ, không cần GCP billing)

**Trước khi chạy:** menu `Runtime -> Change runtime type -> T4 GPU`.

**Chuẩn bị 1 lần:** trên máy Mac, nén project (giữ `artifacts/` để skip G1 đã chạy):
```bash
cd /Users/minhvu/Desktop/Raybanmeta
tar czf uavcl.tgz --exclude='.venv' --exclude='data' uav-continual-learning
```
rồi upload `uavcl.tgz` vào **Google Drive (My Drive)**.

Chạy lần lượt các cell. Bị rớt phiên? Chạy lại từ đầu — run nào xong rồi tự bị skip.

In [ ]:
# 1) Kiểm tra GPU
!nvidia-smi -L

In [ ]:
# 2) Lấy code từ Drive
from google.colab import drive
drive.mount('/content/drive')
!tar xzf /content/drive/MyDrive/uavcl.tgz -C /content/
%cd /content/uav-continual-learning
!ls

In [ ]:
# 3) Cài môi trường (~3 phút; torch đã có sẵn trên Colab)
!pip install -q -r requirements.txt && pip install -q -e . && pip install -q python-docx
!python scripts/check_env.py

In [ ]:
# 4) Toàn bộ test phải xanh trước khi đốt giờ GPU
!python -m pytest -q

In [ ]:
# 5) MỘT LỆNH: đủ mọi giai đoạn G1->G4 + báo cáo docx (EuroSAT, ~2-3h trên T4)
#    Đã upload kèm artifacts/ từ Mac thì G1 tự skip -> nhanh hơn nhiều.
!bash scripts/run_all.sh --quick 2>&1 | tee run_quick.log

In [ ]:
# 6) Lưu kết quả về Drive (docx + toàn bộ metrics + log)
!mkdir -p /content/drive/MyDrive/uavcl_results
!cp artifacts/BAO_CAO_KET_QUA.* /content/drive/MyDrive/uavcl_results/ 2>/dev/null || true
!cp -r artifacts/results /content/drive/MyDrive/uavcl_results/
!cp run_quick.log /content/drive/MyDrive/uavcl_results/ 2>/dev/null || true
print('Xong — mở Drive/uavcl_results/BAO_CAO_KET_QUA.docx')

### (Tuỳ chọn) Bảng chính thức RESISC45 — chạy khi có Colab Pro hoặc phiên dài
```
!bash scripts/run_all.sh 2>&1 | tee run_full.log
```
T4 free có giới hạn phiên (~4h) — RESISC45 full có thể phải chạy 2–3 phiên,
mỗi lần cứ chạy lại cell 5/6: các run xong tự skip, chạy tiếp phần dở.